# Across-days performance for ONE animal

Point this at the **main data directory on the server** and it works through the sessions in order,
asking you for only three things — each one *after* showing you what you need to answer it:

| you set | after seeing |
|---|---|
| **1** the main directory | — |
| **3** the animal | the list of animals found there |
| **6** the protocol and world to analyse | the census of what protocols and worlds are actually in that animal's folder |

Nothing is pre-declared. The steps in between just read and report: list the animals, list that
animal's sessions, read each `log.json` for **date / world / protocol**, and show a
**world × protocol** census. Only then do you choose what to analyse.

The stages are separate on purpose. A mount that is empty, mis-typed or half-synced looks obviously
wrong at step 2 or 4, instead of turning up much later as "0 sessions scored".

Everything comes from the logs — no video, no optic flow — so a month of sessions reads in seconds.

Three things are read from the log and **none is guessed**: the **date**
(`experiment_data.datetime` — sessions are separable by date, folder names are not reliable), the
**animal** (`experiment_data.ID`, normalised to its digits, since the rig writes it free-form:
`mice 168`, `JPASS_0231`), and the **protocol** (the collection effects present).

⚠️ **`view_scale` is the one number NOT in the log** — it is set on a slider in the game UI and never
recorded. It is a property of the **world**, so it is keyed on the world's signature and stated once
per world. A session on an unknown world is marked **unscorable rather than given a default**.


## 1 — where is the data?  ← YOU SET THIS

The only thing that has to be known up front is the **main directory**. The animal is chosen in
step 2, once its folders are listed, and the protocol/world in step 3, once we can see what is
actually there.

In [ ]:
# ===================== STEP 1 of 3: where is the data? =====================
# The MAIN data directory on the server -- the one that holds ALL the animals.
MAIN_DIR = '/mnt/server/data'

PIPELINE_DIR = None       # leave None to locate session_pipeline/ automatically
# ===========================================================================
# ANIMAL is chosen in step 2 (after listing the animals), and the PROTOCOL/WORLD
# to analyse in step 3 (after seeing what protocols and worlds are actually there).

import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

MAIN_DIR = Path(MAIN_DIR).expanduser()
cands = ([Path(PIPELINE_DIR)] if PIPELINE_DIR else []) + [
    Path.cwd(), Path.cwd().parent / 'session_pipeline',
    MAIN_DIR / 'session_pipeline', MAIN_DIR.parent / 'session_pipeline']
PIPE = next((c.resolve() for c in cands if (c / 'common' / 'session_index.py').exists()), None)
if PIPE is None:
    raise FileNotFoundError('could not find session_pipeline/ -- set PIPELINE_DIR. Tried: '
                            + ', '.join(str(c) for c in cands))
sys.path.insert(0, str(PIPE / 'common'))
import perf_from_log as pfl
import session_index as sidx

print(f'pipeline : {PIPE}')
print(f'main dir : {MAIN_DIR}')

## 2 — which animals are in there?

Only folders that actually contain sessions are listed, so an animal is distinguishable from an
unrelated directory sitting next to them. Use this to set `ANIMAL` above.

In [ ]:
ANIMALS = sidx.list_animals(MAIN_DIR)
ANIMALS

## 3 — pick the animal  ← YOU SET THIS

Copy a name from the table above into `ANIMAL` and run on.

In [ ]:
ANIMAL = 'JPAS_168'          # <-- from the `animal` column above

## 4 — list that animal’s sessions

No logs are read yet. This only answers: **is the directory reachable, and what is in it?**
Folders without a `log.json` are named, so a partial copy or a stray folder is obvious.

In [ ]:
ANIMAL_DIR = MAIN_DIR / ANIMAL
if not ANIMAL_DIR.exists():
    raise FileNotFoundError(
        f'no such animal folder: {ANIMAL_DIR}\navailable: {list(ANIMALS.animal)}')

LST = sidx.list_sessions(ANIMAL_DIR)
LST

## 5 — read the logs: which date, which world, which protocol

Now each `log.json` is opened. Three things are taken from it and **none of them is guessed**:

| | read from | why not guessed |
|---|---|---|
| **date** | `experiment_data.datetime` | sessions are separable by date; folder names are not reliable |
| **animal** | `experiment_data.ID`, normalised to its digits | the rig writes it free-form (`mice 168`, `JPASS_0231`) |
| **protocol** | the collection effects actually present | a folder can be named anything |

The **world** (`WxH/iconN/texture.png`) also comes from the log, and it matters because it fixes the
**viewport** — which decides which icons counted as available, which sets the chance baseline.

⚠️ **`view_scale` is the one number NOT in the log.** The logs were searched exhaustively; the only
fractional values anywhere are `version`, autocentre coordinates and direction unit-vectors. It is
set on a slider in the game UI and never recorded. So it is keyed on the **world signature** — stated
once per world, then applied automatically to every session recorded on that world. A session on an
unknown world is marked **unscorable rather than given a default**: the same animal on a new world
must not quietly inherit the old zoom.

Nothing is dropped silently — an unusable session keeps a row and a reason.

In [ ]:
VIEW_SCALES = {}      # filled in below if the census says a world has no scale

S = sidx.discover(ANIMAL_DIR, view_scales=VIEW_SCALES)
if not len(S):
    raise FileNotFoundError(f'no log.json found under {ANIMAL_DIR}')

animal = sidx.require_single_animal(S)      # raises if the folder holds more than one mouse
print(f'animal: {ANIMAL}  (normalised id {animal})   |   {len(S)} session(s)   '
      f'|   {S.day.min()} .. {S.day.max()}')

dupday = S.day.duplicated(keep=False)
if dupday.any():
    print(f'\n{int(dupday.sum())} session(s) share a DAY with another -- labelled a/b in time order:')
    print(S.loc[dupday, ['name', 'day', 'time', 'label']].to_string(index=False))

# ONE SESSION PER DAY. A learning curve's x-axis is the training DAY, so a day holding two
# recordings would contribute two points and be weighted twice. Maryam's rule for this animal:
# KEEP THE LAST session of such a day. Set ONE_PER_DAY = None to keep every session instead.
# The de-selected rows stay in the table with use=False and a note -- nothing disappears.
ONE_PER_DAY = 'last'          # 'last' | 'first' | None
if ONE_PER_DAY:
    print()
    S = sidx.one_per_day(S, keep=ONE_PER_DAY)

# The animal is read from the log's experiment_data.ID -- but that field is free-form and on some
# sessions the rig wrote '_', which carries no animal number at all. Those fall back to the FOLDER
# NAME, which does carry it. `mouse_src` says which source each row used; check it if the split
# below ever looks wrong.
if 'mouse_src' in S and (S.mouse_src == 'folder').any():
    n = int((S.mouse_src == 'folder').sum())
    print(f'\n{n} session(s) took the animal from the FOLDER NAME (the log ID had no number).')
    print(S.loc[S.mouse_src == 'folder', ['name', 'mouse_raw', 'mouse']].head(8).to_string(index=False))

### What worlds and protocols are in here?

This is the answer to *"what have I actually got?"* — before anything is chosen.

**Worlds** are numbered `W1, W2, …` in order of first appearance so they can be referred to by
number. The number is a label for **this listing only**: the log records a world's size, icon width
and texture, not an index, so `W2` here is not a world id from the rig.

**Protocols** are read from the collection effects:

* **`banish_multiplier`** — `banish` + `unbanish` + `single_reward`, rewards carrying a
  **multiplier** (the streak, 1→4). The banishment/punishment task.
* **`timeout_double`** — `single_reward` + `double_reward` + `timeout`, no multiplier.

The **world × protocol** table matters because a protocol can be run on more than one world, and the
world fixes the viewport — hence the chance baseline. Sessions on different worlds are **not
directly poolable** unless each world has its own `view_scale`.

⚠️ If a world shows `view_scale NONE`, add it to `VIEW_SCALES` in the cell above and re-run that
cell — the key to use is the world string printed here.

In [ ]:
CENSUS = sidx.protocol_census(S)
CENSUS

### What the protocol columns mean

`task` is decided by the effects present in the log:

* **`banish_multiplier`** — `banish` + `unbanish` + `single_reward`, and rewards carry a
  **multiplier** (the streak, 1→4). This is the group we want.
* **`timeout_double`** — `single_reward` + `double_reward` + `timeout`, no multiplier.

`effects`, `has_banish`, `has_multiplier` and `max_multiplier` are the raw evidence behind that
label, printed so a **new variant** shows up as its own line rather than being quietly forced into
one of the two known bins.

## 6 — choose what to analyse  ← YOU SET THIS

Now that the protocols and worlds are visible, pick the group. `GROUP_WORLD` is optional: leave it
`None` to take every world running that protocol, or set it to `'W3'` (or `['W3', 'W4']`) to
restrict to one.

Everything not in the group is summarised so nothing disappears unnoticed.

In [ ]:
# ===================== STEP 3 of 3: what to analyse =====================
GROUP_TASK  = 'banish_multiplier'   # from the PROTOCOLS list above
GROUP_WORLD = None                  # None = all worlds | 'W3' | ['W3', 'W4']
BLOCK_SIZE  = 3                     # sessions pooled per block for the criterion (see below)
# ========================================================================

G, rest = sidx.group_report(S, group_task=GROUP_TASK, world_id=GROUP_WORLD)

## 7 — score the group

Only `use=True` sessions are scored. `D` is the discrimination score
`(observed − chance) / (1 − chance)`: 0 = collecting in proportion to what was on screen, 1 = perfect
hazard avoidance. `p` is against the visibility-weighted baseline; `D_exo`/`p_exo` are the exogenous
cross-check (which icon was nearest at spawn — fixed by the board, so his behaviour cannot move it).
The two baselines **bracket** the answer: trust a trend only if it holds under both.

In [ ]:
R = sidx.score(G)
if not len(R):
    raise RuntimeError('no scorable sessions in the group -- see the notes in stage 2')
print(f'\nscored {len(R)} session(s)')

show = R[['label', 'n_trials', 'pos', 'neg', 'acc', 'chance', 'D', 'D_lo', 'D_hi', 'p',
          'chance_exo', 'D_exo', 'p_exo', 'drops', 'coll_per_min', 'drops_per_min',
          'wall_min']].copy()
for c in ['acc', 'chance', 'D', 'D_lo', 'D_hi', 'chance_exo', 'D_exo']:
    show[c] = show[c].round(3)
for c in ['p', 'p_exo']:
    show[c] = show[c].round(4)
for c in ['coll_per_min', 'drops_per_min', 'wall_min']:
    show[c] = show[c].round(2)
show.style.background_gradient(subset=['D'], cmap='RdYlGn', vmin=-0.5, vmax=0.5)

## The learning curve

**D per session with its 95 % confidence interval**, chance at 0. A point + interval is the honest
display: D is one number per session, not a sample, so a box plot would manufacture a distribution
that does not exist. A session is evidence of discrimination only when its interval clears 0.

The layout adapts to however many sessions there are — with many, tick labels are thinned so they
stay readable, and the figure widens.

In [ ]:
def _xticks(a, R, every=None):
    """Thin the tick labels so a long run of sessions stays readable."""
    n = len(R)
    every = every or max(1, int(np.ceil(n / 12)))
    ix = np.arange(n)
    a.set_xticks(ix)
    a.set_xticklabels([lab if i % every == 0 else '' for i, lab in enumerate(R.label)],
                      fontsize=8, rotation=45, ha='right')

W = max(15, min(26, 1.1 * len(R) + 8))          # widen with the number of sessions
fig, ax = plt.subplots(1, 3, figsize=(W, 4.8))
x = np.arange(len(R))

a = ax[0]
a.axhspan(0, 1.05, color='#27ae60', alpha=0.07); a.axhspan(-1.05, 0, color='#c0392b', alpha=0.07)
a.errorbar(x, R.D, yerr=np.array([(R.D - R.D_lo).to_numpy(), (R.D_hi - R.D).to_numpy()]),
           fmt='o-', ms=7, lw=1.8, capsize=4, color='#2c3e50', ecolor='#7f8c8d',
           label='visibility-weighted')
a.plot(x, R.D_exo, 's--', ms=5, color='#f39c12', label='spawn-geometry (exogenous)')
a.axhline(0, color='k', lw=1.6)
for i, r in R.iterrows():
    if r.p < 0.05:
        a.annotate('*', (i, r.D_hi), ha='center', fontsize=15, color='#27ae60')
_xticks(a, R)
a.set_ylabel('discrimination score D'); a.set_ylim(-1.05, 1.05)
a.set_title(f'LEARNING CURVE: D per session -- {ANIMAL}, {len(R)} sessions\n'
            'point + 95% confidence interval;  * = above chance',
            fontsize=10.5, fontweight='bold')
a.legend(fontsize=8); a.grid(alpha=0.2)

a = ax[1]
a.plot(x, R.chance, 's--', ms=5, color='#f39c12', label='chance (visibility-weighted)')
a.plot(x, R.acc, 'o-', ms=6, color='#2c3e50', label='observed accuracy')
_xticks(a, R)
a.set_ylim(0, 1); a.set_ylabel('P(collected a positive icon)')
a.set_title('WHY RAW ACCURACY IS NOT COMPARABLE\nthe chance level moves between sessions',
            fontsize=10.5, fontweight='bold')
a.legend(fontsize=8); a.grid(alpha=0.2)

a = ax[2]
a.plot(x, R.coll_per_min, 'o-', ms=6, color='#27ae60', label='collections / min')
a.plot(x, R.drops_per_min, 's-', ms=5, color='#e67e22', label='reward drops / min')
_xticks(a, R)
a.set_ylabel('per minute'); a.set_title('THROUGHPUT', fontsize=10.5, fontweight='bold')
a.legend(fontsize=8); a.grid(alpha=0.2)
plt.tight_layout()

## ★ The learning criterion — does he avoid the banishment?

**This is the number to watch across days.** Everything above averages the easy trials in with the
informative ones. This isolates the **conflict trials**: both icon types on screen together *and the
banishment the NEARER one*, so he must **override the pull of the closer icon** to take the reward.

* **`conflict_p`** = P(took the reward | banishment was nearer) → **must RISE if he is learning**
* **`control_p`** = the same when the reward was nearer → the **CONTROL**, should stay high

If **both** rise together he has simply stopped following proximity, which is *not* the same as
recognising the icon. The two must **separate** — the gap is plotted directly.

⚠️ **Read the per-session panel for shape only, and the BLOCK panel for evidence.** A single session
carries only ~10–15 conflict trials, so its interval is roughly ±0.25 — wide enough that a real
change is invisible in it. Pooling `BLOCK_SIZE` consecutive sessions (~45 trials at 3) is the
smallest unit that can show one. Blocks are consecutive in time, so the sequence is still a learning
curve.

**How many sessions are needed?** From a baseline near 0.53, detecting a rise takes about 6 sessions
if he reaches 0.80, 10 if 0.75, 17 if 0.70, 35 if 0.65. The binding constraint is the *conflict-trial
count*, not session length — if this becomes the primary question the lever is experimental (bias
spawns so the banishment is nearer more often), not analytical.

In [ ]:
B = sidx.blocks_of(R, BLOCK_SIZE)
print(f'{len(R)} sessions -> {len(B)} block(s) of up to {BLOCK_SIZE}')
print(f'conflict trials per session: median {R.n_conflict.median():.0f} '
      f'(min {R.n_conflict.min():.0f}, max {R.n_conflict.max():.0f})')
print(f'                 per block : median {B.n_conflict.median():.0f}')

fig, ax = plt.subplots(1, 3, figsize=(max(17, W), 5))

a = ax[0]
a.errorbar(x, R.conflict_p,
           yerr=np.array([(R.conflict_p - R.conflict_lo).to_numpy(),
                          (R.conflict_hi - R.conflict_p).to_numpy()]),
           fmt='o-', ms=6, lw=1.6, capsize=3, color='#e67e22', alpha=0.85,
           label='CONFLICT: banishment nearer  (must RISE)')
a.plot(x, R.agree_p, 's--', ms=5, lw=1.4, color='#95a5a6',
       label='CONTROL: reward nearer  (stays high)')
a.axhline(0.5, color='0.6', ls=':', lw=1.5)
_xticks(a, R)
a.set_ylim(0, 1.05); a.set_ylabel('P(collected the reward)')
a.set_title('PER SESSION -- shape only\nintervals are ~+-0.25 wide; do not read one point',
            fontsize=10.5, fontweight='bold')
a.legend(fontsize=8, loc='lower left'); a.grid(alpha=0.2)

a = ax[1]
xb = np.arange(len(B))
a.errorbar(xb, B.conflict_p,
           yerr=np.array([(B.conflict_p - B.conflict_lo).to_numpy(),
                          (B.conflict_hi - B.conflict_p).to_numpy()]),
           fmt='o-', ms=11, lw=2.6, capsize=6, color='#e67e22',
           label=f'CONFLICT (pooled {BLOCK_SIZE} sessions)')
a.errorbar(xb, B.control_p,
           yerr=np.array([(B.control_p - B.control_lo).to_numpy(),
                          (B.control_hi - B.control_p).to_numpy()]),
           fmt='s--', ms=8, lw=2, capsize=5, color='#95a5a6', label='CONTROL')
a.axhline(0.5, color='0.6', ls=':', lw=1.5)
# one fixed level, not offset from each point -- offsets put some labels on the line
for i, r in B.iterrows():
    a.text(i, 0.06, f'{int(r.k_conflict)}/{int(r.n_conflict)}', ha='center', fontsize=8.5,
           color='#8a4b12', fontweight='bold')
a.set_xticks(xb); a.set_xticklabels(B.label, fontsize=8)
a.set_ylim(0, 1.05); a.set_ylabel('P(collected the reward)')
a.set_title('★ THE CRITERION, POOLED INTO BLOCKS\nthis is the panel to read',
            fontsize=11, fontweight='bold', color='#b35418')
# upper left: the counts sit along the bottom, and both lines run in the 0.5-0.8 band
a.legend(fontsize=8.5, loc='upper left'); a.grid(alpha=0.2)
a.set_facecolor('#fdf1e0')
for sp in a.spines.values():
    sp.set_edgecolor('#e67e22'); sp.set_linewidth(2.2)

a = ax[2]
gap = B.control_p - B.conflict_p
a.bar(xb, gap, 0.55, color=['#c0392b' if g > 0 else '#27ae60' for g in gap], alpha=0.9)
a.axhline(0, color='k', lw=1.4)
a.set_xticks(xb); a.set_xticklabels(B.label, fontsize=8)
a.set_ylabel('control - conflict')
a.set_title('THE GAP, per block\nshrinking = he is overriding proximity to avoid the banishment',
            fontsize=10.5, fontweight='bold')
a.grid(alpha=0.2, axis='y')
plt.tight_layout()

### The tests

Two questions, kept apart:

1. **Did the conflict rate move, first block to last?** Fisher's exact on the pooled counts, with the
   control column as the check — if the control moved too, he changed how he follows proximity
   rather than learning the icon.
2. **Is there a trend across all blocks?** Cochran–Armitage-style Spearman on the block rates.

Both are reported even when not significant; with a handful of blocks, "no detectable change yet" is
the expected answer early in training and is worth stating plainly.

In [ ]:
from scipy.stats import fisher_exact, spearmanr

if len(B) < 2:
    print(f'{len(B)} block(s) -- need >=2. Lower BLOCK_SIZE or add sessions.')
else:
    f, l = B.iloc[0], B.iloc[-1]
    p_conf = fisher_exact([[f.k_conflict, f.n_conflict - f.k_conflict],
                           [l.k_conflict, l.n_conflict - l.k_conflict]])[1]
    p_ctrl = fisher_exact([[f.k_control, f.n_control - f.k_control],
                           [l.k_control, l.n_control - l.k_control]])[1]
    print(f'FIRST block ({f.label.replace(chr(10), " ")}):  '
          f'conflict {f.k_conflict}/{f.n_conflict} = {f.conflict_p:.2f}   '
          f'control {f.k_control}/{f.n_control} = {f.control_p:.2f}')
    print(f'LAST  block ({l.label.replace(chr(10), " ")}):  '
          f'conflict {l.k_conflict}/{l.n_conflict} = {l.conflict_p:.2f}   '
          f'control {l.k_control}/{l.n_control} = {l.control_p:.2f}')
    print(f'\n  CONFLICT first->last  p = {p_conf:.3f}   <- the learning test')
    print(f'  CONTROL  first->last  p = {p_ctrl:.3f}   <- should NOT move much')
    if p_conf < 0.05 and p_ctrl >= 0.05:
        print('  => conflict rate MOVED while the control held: consistent with learning to '
              'avoid the banishment.')
    elif p_conf < 0.05:
        print('  => BOTH moved: he changed how he follows proximity, which is NOT the same as '
              'recognising the icon.')
    else:
        print('  => no detectable change yet.')

    if len(B) >= 3:
        rho, pv = spearmanr(np.arange(len(B)), B.conflict_p)
        if np.isfinite(rho):
            print(f'\n  trend across all {len(B)} blocks: rho={rho:+.2f}  p={pv:.3f}')
        else:
            print('\n  trend: flat across blocks (no variation)')

## Is he getting better overall?

The trend in D and throughput across sessions. The honest limit: the interval on a single session's
D is wide (a proportion over ~40–160 trials), so a real trend usually needs several sessions to
separate from noise — and it is believable only if it holds under **both** baselines.

In [ ]:
if len(R) >= 3:
    for col, lab in [('D', 'visibility-weighted D'), ('D_exo', 'exogenous D'),
                     ('conflict_p', 'conflict rate (the criterion)'),
                     ('acc', 'raw accuracy (not comparable!)'),
                     ('coll_per_min', 'collections/min'),
                     ('drops_per_min', 'reward drops/min')]:
        rho, pv = spearmanr(np.arange(len(R)), R[col])
        if not np.isfinite(rho):
            print(f'  {lab:34s} flat across sessions -- no trend defined'); continue
        trend = 'IMPROVING' if rho > 0 else 'DECLINING'
        sig = '' if pv >= 0.05 else '  *significant*'
        print(f'  {lab:34s} rho={rho:+.2f}  p={pv:.3f}   -> {trend}{sig}')
    print('\n  (a trend is believable only if it holds under BOTH baselines;')
    print('   raw accuracy is listed for contrast -- it moves with the chance level, so a trend')
    print('   in it is NOT evidence on its own)')
else:
    print(f'{len(R)} session(s) -- need >=3 for a trend test. Change so far:')
    for col, lab in [('D', 'D'), ('acc', 'accuracy'), ('conflict_p', 'conflict rate'),
                     ('coll_per_min', 'collections/min')]:
        print(f'  {lab:18s} {R[col].iloc[0]:+.3f} -> {R[col].iloc[-1]:+.3f}')

## Save

Writes the per-session scores and the block table next to the animal's folder, so a later session
can be appended without re-deriving anything.

In [ ]:
out = ANIMAL_DIR / f'performance_{GROUP_TASK}.csv'
R.to_csv(out, index=False)
B.to_csv(ANIMAL_DIR / f'performance_{GROUP_TASK}_blocks.csv', index=False)
print('wrote', out)
print('wrote', ANIMAL_DIR / f'performance_{GROUP_TASK}_blocks.csv')